# Tool 6: Return Autocorrelation Analyzer

## Purpose
Determines whether a product's price is **mean-reverting**, **trending**,
or a **random walk** by analyzing the autocorrelation structure of returns.
This is the key diagnostic that tells you HOW to trade each product.

## Why This Matters
Frankfurt Hedgehogs used exactly this analysis to determine:
- Kelp was a random walk -> pure market making (no directional bets)
- Volcanic Rock had significant negative autocorrelation -> mean-reversion trading
- Squid Ink appeared mean-reverting but the real edge was bot detection

## How to Interpret
- **Autocorrelation at lag-1 is significantly negative** -> Mean-reverting.
  Price overshoots, then corrects. Strategy: trade against moves.
- **Autocorrelation at lag-1 is near zero** (within confidence bands) -> Random walk.
  Price is unpredictable. Strategy: pure market making.
- **Autocorrelation at lag-1 is significantly positive** -> Trending/momentum.
  Price continues in the same direction. Strategy: trend-following.

## Monte Carlo Confidence Bands
We generate random walk sequences of the same length and compute their
autocorrelations. The gray bands show the 95% range for a true random walk.
If the actual autocorrelation falls outside these bands, the signal is
statistically significant.

## Dependencies
- `data_loader.py`, `matplotlib`, `pandas`, `numpy`

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

PRODUCT = "ASH_COATED_OSMIUM"  # or "INTARIAN_PEPPER_ROOT"
DAY = -1

# Return horizon: compute returns over this many timesteps
# 1 = single-step returns, 5 = 5-step returns, etc.
RETURN_HORIZONS = [1, 2, 5, 10]

# Maximum lag for autocorrelation computation
MAX_LAG = 20

# Number of Monte Carlo random walk simulations for confidence bands
N_SIMULATIONS = 500

FIG_WIDTH = 18
FIG_HEIGHT = 5

In [ ]:
# ============================================================
# SETUP
# ============================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_prices, compute_wall_mid, AVAILABLE_DAYS, PRODUCTS

prices = compute_wall_mid(load_prices(day=DAY, product=PRODUCT))
mid = prices["wall_mid"].dropna().values

print(f"Loaded {len(mid)} price points for {PRODUCT} on Day {DAY}")
print(f"Price range: {mid.min():.1f} — {mid.max():.1f}")

In [ ]:
# ============================================================
# HELPER: Compute autocorrelation at multiple lags
# ============================================================

def autocorrelation(series, max_lag):
    """Compute autocorrelation of a series at lags 1..max_lag."""
    n = len(series)
    mean = np.mean(series)
    var = np.var(series)
    if var == 0:
        return np.zeros(max_lag)
    ac = np.array([
        np.mean((series[:n-lag] - mean) * (series[lag:] - mean)) / var
        for lag in range(1, max_lag + 1)
    ])
    return ac

In [ ]:
# ============================================================
# PLOT 1: Autocorrelation at Different Return Horizons
# ============================================================
# Each subplot shows autocorrelation for returns computed over a
# different horizon (1-step, 2-step, etc.). Gray bands are 95%
# confidence intervals from random walk Monte Carlo simulations.

n_horizons = len(RETURN_HORIZONS)
fig, axes = plt.subplots(1, n_horizons, figsize=(FIG_WIDTH, FIG_HEIGHT), sharey=True)
if n_horizons == 1:
    axes = [axes]

for h_idx, horizon in enumerate(RETURN_HORIZONS):
    ax = axes[h_idx]
    returns = mid[horizon:] - mid[:-horizon]

    # Actual autocorrelation
    ac = autocorrelation(returns, MAX_LAG)

    # Monte Carlo confidence bands
    mc_acs = np.zeros((N_SIMULATIONS, MAX_LAG))
    for sim in range(N_SIMULATIONS):
        rw = np.cumsum(np.random.randn(len(mid)) * np.std(np.diff(mid)))
        rw_returns = rw[horizon:] - rw[:-horizon]
        mc_acs[sim] = autocorrelation(rw_returns, MAX_LAG)

    lower = np.percentile(mc_acs, 2.5, axis=0)
    upper = np.percentile(mc_acs, 97.5, axis=0)

    lags = np.arange(1, MAX_LAG + 1)
    ax.fill_between(lags, lower, upper, alpha=0.3, color="gray", label="95% CI (random walk)")
    ax.bar(lags, ac, width=0.6, alpha=0.7, color="tab:blue", label="Actual")
    ax.axhline(0, color="black", linewidth=0.5)

    ax.set_xlabel("Lag")
    if h_idx == 0:
        ax.set_ylabel("Autocorrelation")
    ax.set_title(f"Horizon = {horizon}", fontweight="bold")
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(True, alpha=0.3, axis="y")

fig.suptitle(f"Return Autocorrelation — {PRODUCT} (Day {DAY})", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Print lag-1 autocorrelation for quick reference
for horizon in RETURN_HORIZONS:
    returns = mid[horizon:] - mid[:-horizon]
    ac1 = autocorrelation(returns, 1)[0]
    print(f"  Horizon {horizon}: lag-1 autocorrelation = {ac1:.4f}")

In [ ]:
# ============================================================
# PLOT 2: Rolling Autocorrelation Over Time
# ============================================================
# Shows how lag-1 autocorrelation changes through the day.
# If it's consistently negative, mean reversion is persistent.
# If it fluctuates, the signal may only be exploitable at certain times.

ROLLING_WINDOW = 200  # number of returns in the rolling window

returns_1 = pd.Series(mid[1:] - mid[:-1])
rolling_ac = returns_1.rolling(ROLLING_WINDOW).apply(
    lambda x: x.autocorr(lag=1) if len(x) > 1 else np.nan, raw=False
)

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))
ax.plot(prices["timestamp"].values[1:], rolling_ac.values, linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.5)
ax.axhline(-1/np.sqrt(ROLLING_WINDOW), color="gray", linestyle="--", alpha=0.5, label="±1/√N")
ax.axhline(1/np.sqrt(ROLLING_WINDOW), color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Rolling Lag-1 Autocorrelation")
ax.set_title(f"Rolling Autocorrelation (window={ROLLING_WINDOW}) — {PRODUCT} (Day {DAY})", fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# COMPARISON: Both Products
# ============================================================

print("Lag-1 autocorrelation summary (horizon=1):\n")
for prod in PRODUCTS:
    for d in AVAILABLE_DAYS:
        p = compute_wall_mid(load_prices(day=d, product=prod))
        m = p["wall_mid"].dropna().values
        ret = m[1:] - m[:-1]
        ac1 = autocorrelation(ret, 1)[0]
        classification = (
            "MEAN-REVERTING" if ac1 < -0.05 else
            "TRENDING" if ac1 > 0.05 else
            "RANDOM WALK"
        )
        print(f"  {prod} Day {d}: ac1 = {ac1:+.4f}  ->  {classification}")